## Model 3: generating the DBQA data for $^{167}$Er$^{3+}$:Y$_2$SiO$_5$ (site 2) over a field sweep

**Question:** can DBQA resolve the hyperfine levels of the lowest crystal-field level Z1 inside the full 240-state Hamiltonian $H(\vec B)$, accurately enough to locate a ZEFOZ point, and how much of the work carries over from one field to the next?

The Hamiltonian is validated in `Example_8_validation_Er_YSO.ipynb`. That notebook also chose the field sweep: the ZEFOZ point $\vec B^*$ of the crystal-field model with the smallest curvature among those with $0.1\ {\rm T} \le |\vec B^*| \le 1$ T, with the field varied along its direction, $\vec B = b\,\hat n$, over $\pm20\,\%$ of $|\vec B^*|$. Along that line the transition frequency $\nu_{ij}(b)$ has a turning point at $b = |\vec B^*|$.

**Strategy.** Three scales separate: 6 500 cm$^{-1}$ between the multiplets, 27 cm$^{-1}$ from Z1 to Z2, and hyperfine gaps of 0.01–0.1 cm$^{-1}$ inside Z1. As in Model 2, a first DBQA stage isolates the target block and a second stage diagonalises it:
- **Stage A** (8 qubits, energies in units of 5 000 cm$^{-1}$): group-commutator DBQA on the full embedded Hamiltonian until the 16 basis states with the lowest diagonal entries span Z1 to $1-F < \epsilon$, with $\epsilon = 10^{-3}, 10^{-6}, 10^{-9}, 10^{-12}$.
- **Stage B** (4 qubits): DBQA on that $16\times16$ block, rescaled to unit norm, until the hyperfine levels are resolved.

The duration grid must reach the inverse of the smallest gap to be resolved in the rescaled units (the Z1–Z2 gap is $5.4\times10^{-3}$ in units of 5 000 cm$^{-1}$). We use $s \in [10^{-3}, 10^2]$ in stage A, against $s \le 10$ in the Model 2 notebooks.

**Three ways to start each field point** (stage A):
1. *cold*: from $H(b)$;
2. *warm*: from $U_{\rm prev}^\dagger H(b)\,U_{\rm prev}$, where $U_{\rm prev}$ is the total stage-A rotation reached at the previous field ($1-F<10^{-12}$);
3. *replay*: the previous field's schedule of durations, applied to $H(b)$ with $D = \Delta(\hat H_k)$ recomputed at every step, then grid search as in the cold start.

On a quantum computer the warm start is not free: $U_{\rm prev}$ is itself a product of evolutions under the previous Hamiltonians, so the circuit keeps all earlier steps. The replay keeps the circuit length of a cold start and saves only the classical search for the durations. We report all three.

Runs are deterministic: grid-search scheduling has no random element. The cold starts take about 45 minutes on 8 processes and the sequential warm and replay chains about 40 minutes; the notebook took 84 minutes with another notebook running alongside.

In [1]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"

import logging
import multiprocessing as mp
import pickle
import time
from concurrent.futures import ProcessPoolExecutor

import numpy as np

from qibo import set_backend
from qibo.config import log

from pycf.import_sljm import ImportSLJM

from qibo_dbqa.double_bracket import DoubleBracketGeneratorType, DoubleBracketIteration
from qibo_dbqa.utils import generate_Z_operators
from qibo_dbqa.utils_crystal_field import *

os.environ.setdefault("PYCF_ROOT", os.path.abspath("../../../pycf"))
set_backend("numpy")
log.setLevel(logging.ERROR)
os.makedirs("simulation_results", exist_ok=True)

/home/aarush_das_bansiwal/micromamba/envs/dbqa-cf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Qibo 0.2.14|INFO|2026-10-17 07:22:07]: Using numpy backend on /CPU:0


In [2]:
with open("simulation_results/Er_YSO_validation.pickle", "rb") as file:
    validation = pickle.load(file)
sweep = validation["sweep"]
i, j = sweep["transition"]
direction = sweep["direction"]
magnitudes = sweep["magnitudes"]
print(f"ZEFOZ point of |{i}>-|{j}>: B* = {np.round(sweep['field'] * 1e3, 1)} mT, |B*| = {np.linalg.norm(sweep['field']) * 1e3:.1f} mT, nu = {sweep['frequency']:.3f} MHz")
print(f"{len(magnitudes)} field points from {magnitudes[0] * 1e3:.1f} to {magnitudes[-1] * 1e3:.1f} mT")

erhfs = ImportSLJM(pycf_matel_path("erhfs", "eryso"))
model = hyperfine_field_model(erhfs, er_yso_pycf_coeff(hyperfine=ER_YSO_SITE2_HYPERFINE_JOBBITT2021))
operators = field_operators(model)

ENERGY_UNIT = 5000.0  # cm^-1


def embedded(b):
    # traceless embedded Hamiltonian at field b * direction, in units of ENERGY_UNIT
    padded, _, _ = embed_in_qubits(linear_field_hamiltonian(model["h0"], operators, b * direction))
    return (padded - np.trace(padded).real / len(padded) * np.eye(len(padded))) / ENERGY_UNIT


exact = np.array([np.linalg.eigvalsh(linear_field_hamiltonian(model["h0"], operators, b * direction))[:16] for b in magnitudes])
gap_z1_z2 = min(np.linalg.eigvalsh(linear_field_hamiltonian(model["h0"], operators, b * direction))[16] - e[15] for b, e in zip(magnitudes, exact))
print(f"smallest Z1-Z2 gap over the sweep: {gap_z1_z2:.2f} cm^-1; smallest hyperfine gap inside Z1: {np.min(np.diff(exact, axis=1)) * CM_TO_MHZ:.1f} MHz")

ZEFOZ point of |6>-|7>: B* = [-148.2    8.3   20.8] mT, |B*| = 149.9 mT, nu = 766.822 MHz
21 field points from 119.9 to 179.8 mT


smallest Z1-Z2 gap over the sweep: 28.34 cm^-1; smallest hyperfine gap inside Z1: 761.7 MHz


#### Stage A

`stark` groups the sorted spectrum into the 15 crystal-field levels of 16 hyperfine states and the 16 padding states. This is valid while the Zeeman splitting stays below the gaps between crystal-field levels; the previous cell checks that Z1 and Z2 remain separated by more than 25 cm$^{-1}$. A report is also recorded for the two multiplets.

In [3]:
STARK = [list(range(16 * k, 16 * k + 16)) for k in range(16)]
MULTIPLETS = [list(range(128)), list(range(128, 240)), list(range(240, 256))]
THRESHOLDS = (1e-3, 1e-6, 1e-9, 1e-12)
SPACE_A = np.logspace(-3, 2, 100)
NSTEPS_A = 250


def lowest_block(matrix, size=16):
    index = np.sort(np.argsort(np.real(np.diag(matrix)))[:size])
    return matrix[np.ix_(index, index)]


def summarise(run, k_field):
    # iterations, Z1 transition-frequency errors (exact diagonalisation of the block) and blocks at every threshold
    reference = (exact[k_field] - exact[k_field][0]) * CM_TO_MHZ
    out = dict(steps=run["steps"], history=[dict(stark=r["stark"]["level_fidelity"][0], multiplets=r["multiplets"]["level_fidelity"][0]) for r in run["history"]], captured={})
    for threshold, (k, matrix) in run["captured"].items():
        block = lowest_block(matrix) * ENERGY_UNIT
        e = np.linalg.eigvalsh(block)
        out["captured"][threshold] = dict(iteration=k, block=block, transition_error=np.max(np.abs((e - e[0]) * CM_TO_MHZ - reference)))
    return out


def cold_run(k_field):
    run = run_dbqa_history(embedded(magnitudes[k_field]), "group commutator", dict(stark=STARK, multiplets=MULTIPLETS), NSTEPS_A, SPACE_A,
                           capture=("stark", 0, THRESHOLDS), track_unitary=True, stop_when_captured=True)
    return summarise(run, k_field), run["captured_unitary"].get(1e-12)


start = time.time()
with ProcessPoolExecutor(max_workers=min(len(magnitudes), 8), mp_context=mp.get_context("fork")) as pool:
    cold_results = list(pool.map(cold_run, range(len(magnitudes))))
cold = [result for result, _ in cold_results]
print(f"cold starts: {(time.time() - start) / 60:.1f} min")
for b, result in zip(magnitudes, cold):
    print(f"   {b * 1e3:7.1f} mT: " + ", ".join(f"1-F<{t:g} at {c['iteration']} ({c['transition_error']:.1e} MHz)" for t, c in sorted(result["captured"].items(), reverse=True)))

cold starts: 46.2 min
     119.9 mT: 1-F<0.001 at 54 (3.2e+02 MHz), 1-F<1e-06 at 79 (4.1e-01 MHz), 1-F<1e-09 at 108 (9.7e-05 MHz), 1-F<1e-12 at 131 (9.1e-06 MHz)
     122.9 mT: 1-F<0.001 at 54 (3.2e+02 MHz), 1-F<1e-06 at 79 (4.0e-01 MHz), 1-F<1e-09 at 108 (8.4e-05 MHz), 1-F<1e-12 at 130 (7.0e-06 MHz)
     125.9 mT: 1-F<0.001 at 54 (3.2e+02 MHz), 1-F<1e-06 at 79 (3.9e-01 MHz), 1-F<1e-09 at 108 (7.0e-05 MHz), 1-F<1e-12 at 130 (1.6e-05 MHz)
     128.9 mT: 1-F<0.001 at 54 (3.2e+02 MHz), 1-F<1e-06 at 80 (3.7e-01 MHz), 1-F<1e-09 at 108 (7.7e-05 MHz), 1-F<1e-12 at 132 (1.2e-05 MHz)
     131.9 mT: 1-F<0.001 at 54 (3.2e+02 MHz), 1-F<1e-06 at 80 (3.6e-01 MHz), 1-F<1e-09 at 105 (1.1e-04 MHz), 1-F<1e-12 at 138 (8.6e-06 MHz)
     134.9 mT: 1-F<0.001 at 54 (3.1e+02 MHz), 1-F<1e-06 at 81 (3.3e-01 MHz), 1-F<1e-09 at 104 (8.7e-05 MHz), 1-F<1e-12 at 128 (2.2e-05 MHz)
     137.9 mT: 1-F<0.001 at 54 (3.1e+02 MHz), 1-F<1e-06 at 81 (3.2e-01 MHz), 1-F<1e-09 at 104 (8.6e-05 MHz), 1-F<1e-12 at 133 (2.8e-05 MHz

In [4]:
def replay(matrix, schedule):
    # apply a fixed schedule of group-commutator durations, D = Delta(H_k) at every step
    dbi = DoubleBracketIteration(Hamiltonian(8, matrix), mode=DoubleBracketGeneratorType.group_commutator)
    unitary = np.eye(len(matrix), dtype=complex)
    for step in schedule:
        unitary = unitary @ dbi(step=step)
    return dbi.h.matrix, unitary


warm, replayed = [None] * len(magnitudes), [None] * len(magnitudes)
start = time.time()
u_prev = cold_results[0][1]
schedule_prev = cold[0]["steps"]
for k_field in range(1, len(magnitudes)):
    h = embedded(magnitudes[k_field])
    run = run_dbqa_history(u_prev.conj().T @ h @ u_prev, "group commutator", dict(stark=STARK, multiplets=MULTIPLETS), NSTEPS_A, SPACE_A,
                           capture=("stark", 0, THRESHOLDS), track_unitary=True, stop_when_captured=True)
    warm[k_field] = summarise(run, k_field)
    if 1e-12 in run["captured_unitary"]:
        u_prev = u_prev @ run["captured_unitary"][1e-12]
    replay_matrix, _ = replay(h, schedule_prev)
    run = run_dbqa_history(replay_matrix, "group commutator", dict(stark=STARK, multiplets=MULTIPLETS), NSTEPS_A, SPACE_A,
                           capture=("stark", 0, THRESHOLDS), stop_when_captured=True)
    replayed[k_field] = summarise(run, k_field)
    replayed[k_field]["replayed_steps"] = len(schedule_prev)
    schedule_prev = cold[k_field]["steps"]
    print(f"{magnitudes[k_field] * 1e3:7.1f} mT  warm: " + ", ".join(f"{t:g}@{c['iteration']}" for t, c in sorted(warm[k_field]["captured"].items(), reverse=True))
          + "  | replay of " + f"{replayed[k_field]['replayed_steps']} steps, then: " + ", ".join(f"{t:g}@{c['iteration']}" for t, c in sorted(replayed[k_field]["captured"].items(), reverse=True)))
print(f"warm and replay chains: {(time.time() - start) / 60:.1f} min")

  122.9 mT  warm: 0.001@0, 1e-06@0, 1e-09@9, 1e-12@51  | replay of 131 steps, then: 0.001@0, 1e-06@0, 1e-09@0, 1e-12@0


  125.9 mT  warm: 0.001@0, 1e-06@0, 1e-09@8, 1e-12@72  | replay of 130 steps, then: 0.001@0, 1e-06@0, 1e-09@0, 1e-12@0


  128.9 mT  warm: 0.001@0, 1e-06@0, 1e-09@9, 1e-12@36  | replay of 130 steps, then: 0.001@0, 1e-06@0, 1e-09@0, 1e-12@0


  131.9 mT  warm: 0.001@0, 1e-06@0, 1e-09@8, 1e-12@39  | replay of 132 steps, then: 0.001@0, 1e-06@0, 1e-09@0, 1e-12@0


  134.9 mT  warm: 0.001@0, 1e-06@0, 1e-09@9, 1e-12@49  | replay of 138 steps, then: 0.001@0, 1e-06@0, 1e-09@0, 1e-12@0


...

  167.8 mT  warm: 0.001@0, 1e-06@0, 1e-09@7, 1e-12@59  | replay of 151 steps, then: 0.001@0, 1e-06@0, 1e-09@0, 1e-12@0


  170.8 mT  warm: 0.001@0, 1e-06@0, 1e-09@7, 1e-12@44  | replay of 135 steps, then: 0.001@0, 1e-06@0, 1e-09@0, 1e-12@0


  173.8 mT  warm: 0.001@0, 1e-06@0, 1e-09@9, 1e-12@41  | replay of 143 steps, then: 0.001@0, 1e-06@0, 1e-09@0, 1e-12@0


  176.8 mT  warm: 0.001@0, 1e-06@0, 1e-09@8, 1e-12@152  | replay of 218 steps, then: 0.001@0, 1e-06@0, 1e-09@0, 1e-12@0


  179.8 mT  warm: 0.001@0, 1e-06@0, 1e-09@7, 1e-12@40  | replay of 247 steps, then: 0.001@0, 1e-06@0, 1e-09@0, 1e-12@0
warm and replay chains: 37.0 min


#### Stage B

The $16\times16$ block captured at $1-F<\epsilon$ is shifted to zero trace and divided by its spectral norm. Group-commutator, canonical and Pauli-Z DBQA (with the $2^4-1$ products of Z operators as candidate $D$) then diagonalise it. The DBQA estimates of the hyperfine levels are the sorted diagonal entries. We record the error of all 120 transition frequencies of Z1 and of the ZEFOZ transition $\nu_{ij}$, every iteration, from the blocks captured at $\epsilon = 10^{-6}, 10^{-9}, 10^{-12}$ (cold starts). Stage B is also warm-started along the sweep: the block rotation of the previous field point is applied first.

In [5]:
NSTEPS_B = 60
SPACES_B = {"group commutator": np.logspace(-3, 3, 120), "canonical": np.logspace(-3, 3, 120), "Pauli-Z": np.logspace(-3, 3, 40)}
Z_OPS = [np.real(op) for op in generate_Z_operators(4).values()]
SINGLES = [[k] for k in range(16)]


def stage_b(block, k_field, method, rotation=None):
    shifted = block - np.trace(block).real / 16 * np.eye(16)
    unit = np.max(np.abs(np.linalg.eigvalsh(shifted)))
    matrix = shifted / unit if rotation is None else rotation.conj().T @ (shifted / unit) @ rotation
    run = run_dbqa_history(matrix, method, dict(levels=SINGLES), NSTEPS_B, SPACES_B[method], d_list=Z_OPS if method == "Pauli-Z" else None,
                           checkpoints=tuple(range(NSTEPS_B + 1)), track_unitary=True)
    reference = (exact[k_field] - exact[k_field][0]) * CM_TO_MHZ
    transition_error, zefoz_error, nu = [], [], []
    for k in range(NSTEPS_B + 1):
        d = np.sort(np.real(np.diag(run["checkpoints"][k]))) * unit
        estimate = (d - d[0]) * CM_TO_MHZ
        transition_error.append(np.max(np.abs(estimate[:, None] - estimate[None, :] - (reference[:, None] - reference[None, :]))))
        nu.append(estimate[j] - estimate[i])
        zefoz_error.append(abs(nu[-1] - (reference[j] - reference[i])))
    return dict(unit=unit, transition_error=np.array(transition_error), zefoz_error=np.array(zefoz_error), nu=np.array(nu), unitary=run["unitary"])


def stage_b_field(k_field):
    out = {}
    for threshold in (1e-6, 1e-9, 1e-12):
        if threshold in cold[k_field]["captured"]:
            block = cold[k_field]["captured"][threshold]["block"]
            out[threshold] = {method: stage_b(block, k_field, method) for method in SPACES_B}
            for result in out[threshold].values():
                result.pop("unitary")
    return out


start = time.time()
with ProcessPoolExecutor(max_workers=min(len(magnitudes), 8), mp_context=mp.get_context("fork")) as pool:
    stage2 = list(pool.map(stage_b_field, range(len(magnitudes))))
print(f"stage B, cold: {(time.time() - start) / 60:.1f} min")
k_mid = len(magnitudes) // 2
for threshold, runs in stage2[k_mid].items():
    print(f"|B| = {magnitudes[k_mid] * 1e3:.1f} mT, block from 1-F < {threshold:g}: " + "; ".join(
        f"{method}: max transition error {r['transition_error'][20]:.1e} (k=20), {r['transition_error'][-1]:.1e} MHz (k={NSTEPS_B})" for method, r in runs.items()))

stage B, cold: 0.2 min
|B| = 149.9 mT, block from 1-F < 1e-06: group commutator: max transition error 3.5e-01 (k=20), 3.6e-01 MHz (k=60); canonical: max transition error 4.7e+03 (k=20), 4.6e+03 MHz (k=60); Pauli-Z: max transition error 5.7e+02 (k=20), 7.9e+01 MHz (k=60)
|B| = 149.9 mT, block from 1-F < 1e-09: group commutator: max transition error 4.2e-02 (k=20), 1.7e-04 MHz (k=60); canonical: max transition error 4.7e+03 (k=20), 4.6e+03 MHz (k=60); Pauli-Z: max transition error 5.7e+02 (k=20), 8.5e+01 MHz (k=60)
|B| = 149.9 mT, block from 1-F < 1e-12: group commutator: max transition error 4.2e-02 (k=20), 2.6e-05 MHz (k=60); canonical: max transition error 4.7e+03 (k=20), 4.6e+03 MHz (k=60); Pauli-Z: max transition error 5.7e+02 (k=20), 5.3e+01 MHz (k=60)


In [6]:
# stage B warm-started along the sweep (group commutator, blocks from 1-F < 1e-12)
stage2_warm = [None] * len(magnitudes)
rotation = None
for k_field in range(len(magnitudes)):
    if 1e-12 not in cold[k_field]["captured"]:
        continue
    result = stage_b(cold[k_field]["captured"][1e-12]["block"], k_field, "group commutator", rotation=rotation)
    rotation = result.pop("unitary") if rotation is None else rotation @ result.pop("unitary")
    stage2_warm[k_field] = result
k_warm = [k for k, r in enumerate(stage2_warm) if r is not None]
print("warm stage B, iterations to a 0.01 MHz transition error:",
      [int(np.argmax(stage2_warm[k]["transition_error"] < 1e-2)) if np.any(stage2_warm[k]["transition_error"] < 1e-2) else None for k in k_warm])

warm stage B, iterations to a 0.01 MHz transition error: [23, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 2, 1, 2, 1, 1, 0, 2, 2, 3]


#### Saving the data

In [7]:
with open("simulation_results/Er_YSO_DBQA_L_8.pickle", "wb") as file:
    pickle.dump(
        dict(
            sweep=sweep, energy_unit=ENERGY_UNIT, exact=exact, thresholds=THRESHOLDS, space_a=SPACE_A, nsteps_a=NSTEPS_A,
            nsteps_b=NSTEPS_B, spaces_b=SPACES_B,
            cold=[{k: v for k, v in r.items()} for r in cold], warm=warm, replayed=replayed,
            stage2=stage2, stage2_warm=stage2_warm,
        ),
        file,
    )
print(f"{os.path.getsize('simulation_results/Er_YSO_DBQA_L_8.pickle') / 1e6:.1f} MB")

1.7 MB
